# PoliMillionaire — NLP Chatbot Assignment

A chatbot that plays the *Who wants to be a PoliMillionaire?* quiz via a REST API client.

**Competitions:**
| ID | Name | Questions |
|----|------|-----------|
| 0  | Entertainment | 15 |
| 1  | Ancient History & Politics | 15 |
| 2  | Science & Nature | 15 |
| 3  | Maths | 15 |

**Architecture:**
- **LLM:** Qwen2.5-7B-Instruct (float16, greedy decoding for speed)
- **RAG:** DuckDuckGo/ddgs (Entertainment) · Wikipedia (History) · Multi-query Wikipedia+DuckDuckGo + cross-encoder reranking (Science) · Calculator (Maths)
- **Answer extraction:** regex-based, robust fallback chain (`ANSWER: X` tag → digit → letter)

## 0 · Setup — Clone from GitHub

Clone the repository so `millionaire_client` and `millionaire_bot.py` are available.

In [1]:
!git clone https://github.com/riccardo03/NLP_university_project.git 2>/dev/null || (cd NLP_university_project && git pull)

In [2]:
import sys, os

PACKAGE_DIR = '/content/NLP_university_project'

if PACKAGE_DIR not in sys.path:
    sys.path.append(PACKAGE_DIR)

print('Path configured:', PACKAGE_DIR)

Path configured: /content/NLP_university_project


## 1 · Install dependencies

Required packages beyond the standard Colab environment:

In [3]:
%pip install -q ddgs wikipedia transformers accelerate sentence-transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 72.7 MB/s eta 0:00:00


In [4]:
!pip install faiss-cpu sentence-transformers datasets torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.8 MB/s eta 0:00:00


## 2 · Import the bot module

All game logic lives in `millionaire_bot.py`. Import it, we do.

In [5]:
import millionaire_bot as bot

print('Imported successfully, the bot module has.')

Imported successfully, the bot module has.


## 3 · Load the LLM

Qwen2.5-7B-Instruct is loaded with `device_map="auto"` and `torch_dtype=float16`.
This downloads ~14 GB on first run — patience, the Force requires.

In [6]:
bot.load_model('Qwen/Qwen2.5-Math-7B-Instruct')

Loading model: Qwen/Qwen2.5-Math-7B-Instruct


config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

[Maths RAG] Embedding dimension: 768


In [10]:
!pip install sentence-transformers

In [11]:
!pip install gliner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 95.3 MB/s eta 0:00:00


In [21]:
import sys
import importlib
import subprocess
# import millionaire # Il tuo punto d'ingresso principale <- This line was causing the error

# 1. Aggiornamento da GitHub
print("🔄 Aggiornamento repository in corso...")
subprocess.run(["git", "-C", "/content/NLP_university_project", "reset", "--hard"], check=True)
subprocess.run(["git", "-C", "/content/NLP_university_project", "pull"], check=True)

# 2. Backup dei pesi del modello (assumendo che siano dentro 'bot' che è millionaire_bot)
# Le referenze a 'millionaire.bot' sono state cambiate a 'bot'
_model     = bot._model
_pipe      = bot._pipe
_tokenizer = bot._tokenizer

# 3. Lista specifica dei tuoi file da ricaricare
# L'ordine conta: carichiamo prima i moduli "rag_" e per ultimo il main
moduli_progetto = [
    "rag_maths",
    "rag_science",
    "rag_entertainment",
    "rag_history",
    "millionaire_bot"
]

print("\n🚀 Ricaricamento moduli...")
for nome_modulo in moduli_progetto:
    if nome_modulo in sys.modules:
        try:
            importlib.reload(sys.modules[nome_modulo])
            print(f"✅ {nome_modulo} ricaricato.")
        except Exception as e:
            print(f"❌ Errore durante il reload di {nome_modulo}: {e}")

# 4. Re-iniezione dei pesi
# Dopo il reload, bot (millionaire_bot) è "pulito", quindi ripristiniamo il modello
# Le referenze a 'millionaire.bot' sono state cambiate a 'bot'
bot._model     = _model
bot._pipe      = _pipe
bot._tokenizer = _tokenizer

print("\n✨ Sistema aggiornato! Modello Qwen preservato.")

🔄 Aggiornamento repository in corso...

🚀 Ricaricamento moduli...
✅ rag_maths ricaricato.
✅ rag_science ricaricato.
✅ rag_entertainment ricaricato.
✅ rag_history ricaricato.
✅ millionaire_bot ricaricato.

✨ Sistema aggiornato! Modello Qwen preservato.


## 4 · Connect to the API

Authenticate with the PoliMillionaire server. Store your password in a Colab secret named `poli-millionaire`.

In [7]:
from google.colab import userdata
from millionaire_client import MillionaireClient, AuthenticationError

API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'riccardo'           # Change to your username
PASSWORD = userdata.get('poli-millionaire')

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Welcome, {user.username}! Role: {user.role}')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Welcome, riccardo! Role: student


## 5 · Browse available competitions

In [ ]:
competitions = client.competitions.list_all()
print('=== Available Competitions ===')
for c in competitions:
    print(f'  [{c.id}] {c.name} — {c.max_levels} questions')

=== Available Competitions ===
  [0] Entertainment — 15 questions
  [1] Ancient History and Politics — 15 questions
  [2] Science and Nature — 15 questions
  [3] Maths — 15 questions


## 6 · Play a single competition

Choose a `COMP_ID` (0–3) and let the bot play one full game.

In [22]:
import rag_science # Explicitly import rag_science
import rag_maths

#rag_science.setup_science_rag() # Initialize Science RAG
#rag_maths.setup_maths_rag()    # Initialize Maths RAG

COMP_ID = bot.COMP_MATHS   # Change this to try other competitions

game = client.game.start(competition_id=COMP_ID)
print(f'Session ID: {game.session_id} | Questions: {game.state.competition.max_levels}')

game_log = bot.play_game(game, COMP_ID)
bot.print_evaluation(game_log)

[Maths RAG] Building corpus...
      Loaded AI-MO/NuminaMath-CoT: +80,000 passages
      79,697 passages total
[Maths RAG] Embedding with allenai/specter ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/2491 [00:00<?, ?it/s]

[Maths RAG] Embedding dimension: 768
Session ID: 141251 | Questions: 15

  Starting: Maths

--- Level 1 | Time: 29.9s ---
Q: We roll a fair 6-sided die 5 times. What is the probability that we get a 6 in at most 2 of the rolls?
  [0] \frac{1}{648}
  [1] \frac{125}{648}
  [2] \frac{25}{648}
  [3] \frac{625}{648}
  [RAG] Searching for context...
  [RAG] Done in 0.0s. Context: We roll a fair 6-sided die 6 times. What is the probability that we get an odd number in exactly 3 of the 6 rolls?  A fa...
  [LLM] Thinking...
  [LLM] Output: 'To determine the probability of getting a 6 in at most 2 of the 5 rolls of a fair 6-sided die, we can use the binomial probability formula. The binomial probability' → Answer ID: 2 (in 28.0s)
  ✗ WRONG! Game over. Earned: $0.00

  Maths — Level reached: 1 | Earnings: $0.00


──────────────────────────────────────────────────
  EVALUATION — Maths
──────────────────────────────────────────────────
  Level reached : 1
  Earnings      : $0.00
  Questions     : 1

## 7 · Play all four competitions

Run all competitions sequentially and aggregate results.
A complete evaluation, this produces.

In [ ]:
all_logs = []

for comp_id in [bot.COMP_ENTERTAINMENT,
                bot.COMP_HISTORY_POLITICS,
                bot.COMP_SCIENCE_NATURE,
                bot.COMP_MATHS]:
    game = client.game.start(competition_id=comp_id)
    print(f'Session {game.session_id} started for competition {comp_id}.')
    log = bot.play_game(game, comp_id)
    all_logs.append(log)

bot.print_all_evaluations(all_logs)

Session 58039 started for competition 0.

  Starting: Entertainment

--- Level 1 | Time: 29.9s ---
Q: What was the primary reason James Cameron switched from studying physics to English at Fullerton College?
  [0] He wanted to become a writer
  [1] He was inspired by the success of Star Wars
  [2] He found physics too difficult
  [3] He was offered a job in the film industry
  [RAG] Searching for context...


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [RAG] Done in 1.6s. Context: [James Cameron - Wikipedia] 1 day ago - After high school, Cameron enrolled at Fullerton College, a community college, i...
  [LLM] Thinking...
  [LLM] Output: '0' → Answer ID: 0 (in 1.8s)
  ✗ WRONG! Game over. Earned: $0.00

  Entertainment — Level reached: 1 | Earnings: $0.00

Session 58040 started for competition 1.

  Starting: Ancient History & Politics

--- Level 1 | Time: 29.9s ---
Q: How many ancient biographies of Homer are mentioned in the article?
  [0] One
  [1] Four
  [2] Three
  [3] Two
  [RAG] Searching for context...


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [RAG] Done in 1.2s. Context: Ancient accounts of Homer include numerous passages in which archaic and classical Greek poets and prose authors mention...
  [LLM] Thinking...
  [LLM] Output: '2' → Answer ID: 2 (in 1.6s)
  ✗ WRONG! Game over. Earned: $0.00

  Ancient History & Politics — Level reached: 1 | Earnings: $0.00



...

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [RAG-Science] Stage2 retrieval: 3.6s → 5 snippets
  [RAG-Science] Stage3 reranking: 0.0s
  [RAG] Done in 19.1s. Context: [Soil erosion - Wikipedia] Valley or stream erosion occurs with continued water flow along a linear feature. The erosion...
  [LLM] Thinking...
  [LLM] Output: '1' → Answer ID: 1 (in 2.1s)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✓ CORRECT! Earned so far: $100.00

--- Level 2 | Time: 29.9s ---
Q: Which condition is necessary for metamorphic rocks to form?
  [0] slow cooling of magma
  [1] constant weathering and erosion
  [2] extreme pressure and heating
  [3] rapid burial of sediments
  [RAG] Searching for context...


KeyboardInterrupt: 

## 8 · Leaderboard

See where the bot ranks after playing.

In [11]:
for comp_id in range(4):
    lb = client.leaderboard.get(competition_id=comp_id, limit=50)
    print(f'\n=== Leaderboard: {lb.competition.name} ===')
    for i, entry in enumerate(lb.entries[:50], 1):
        marker = ' <-- YOU' if entry.username == USERNAME else ''
        print(f'  {i:>2}. {entry.username:<20} ${entry.score:>10,.2f}  (Level {entry.reached_level}){marker}')


=== Leaderboard: Entertainment ===
   1. ialkbr               $1,024,000.00  (Level 15)
   2. Zero37               $1,024,000.00  (Level 15)
   3. supreme_leader       $1,024,000.00  (Level 15)
   4. luca_bordin          $1,024,000.00  (Level 15)
   5. Anonymous            $1,024,000.00  (Level 15)
   6. Jasmin               $1,024,000.00  (Level 15)
   7. AleAssini            $1,024,000.00  (Level 15)
   8. MatteoVitali         $1,024,000.00  (Level 15)
   9. TheLastGuessbender   $1,024,000.00  (Level 15)
  10. lillaro              $1,024,000.00  (Level 15)
  11. gab                  $1,024,000.00  (Level 15)
  12. SaturdaY             $1,024,000.00  (Level 15)
  13. mahdisiami           $1,024,000.00  (Level 15)
  14. Stanino              $1,024,000.00  (Level 15)
  15. Bilal_jaiel          $1,024,000.00  (Level 15)
  16. vpetrak              $1,024,000.00  (Level 15)
  17. FrancescoMazzola     $1,024,000.00  (Level 15)
  18. samin                $1,024,000.00  (Level 15)
  19. Andr

## 9 · Quick unit tests for key functions

Verify the helper functions work correctly before running a live game.

In [ ]:
# --- Test extract_answer_id ---
cases = [
    ('The answer is 2, definitely.',          0, 4, 2),
    ('I think option B is correct.',          0, 4, 1),
    ('No clue at all.',                       0, 4, 0),
    ('3_Full chain-of-thought reasoning...',  0, 4, 3),
    ('Option A seems right.',                 0, 4, 0),
]
print('=== extract_answer_id tests ===')
all_passed = True
for text, default, n_opts, expected in cases:
    got = bot.extract_answer_id(text, n_opts)
    status = '✓' if got == expected else '✗'
    if got != expected:
        all_passed = False
    print(f'  {status}  "{text[:40]}" → {got} (expected {expected})')
print('All passed!' if all_passed else 'Some failed!')

# --- Test rag_maths ---
print('\n=== rag_maths tests ===')
maths_cases = [
    'What is 15% of 200?',
    'Calculate the square root of 144.',
    'What is 2^10?',
    'What is 3 + 4 * 2?',
]
for q in maths_cases:
    result = bot.rag_maths(q)
    print(f'  Q: {q}')
    print(f'  → {result}\n')

## 10 · Save logs to JSON

Persist all game logs for offline analysis.

In [ ]:
import json

log_path = '/content/game_logs.json'
with open(log_path, 'w') as f:
    json.dump(all_logs, f, indent=2)

print(f'Saved {len(all_logs)} game log(s) to {log_path}')